This notebook identifies the subset of "required metabolites" that act as substrates in the E-matrix. The production of such metabolites should be protected during context-extraction.

In [18]:
import os
import urllib
import json
import re
from tqdm import tqdm
import copy

import pandas as pd
import numpy as np
import cobra
import sympy

import seaborn as sns
import matplotlib.pyplot as plt

from human_me import io
from human_me.data.file_paths import input_local_path, build_files_url, build_local_path
from human_me.preprocess.correct_inputs import correct_model, correct_psim
from human_me.utils import parameters as params
from human_me.utils.functions import flatten_list
from human_me.core.biomass import check_m_biomass
from human_me.build.build_me_model import build_me

In [19]:
n_cores = 20
human_me_data_path = '/data3/hratch/human_me_data/'

Load the model and apply standard preprocessing:

In [21]:
recon2 = io.load_metabolic_model(os.path.join(input_local_path, 'recon2_2.xml'))
_, cm_2, me_input_model = correct_model(recon2.copy(), 
                           correct_biomass = True
                          )
psim_me, _, _ = correct_psim(me_input_model)

/data2/hratch/Software/me_analyses_1_env/lib/python3.9/site-packages/human_me/preprocess/correct_inputs.py:121 UserWarning: ACCOAC contains redundant complexes according to GPR, editing GPR
/data2/hratch/Software/me_analyses_1_env/lib/python3.9/site-packages/human_me/preprocess/correct_inputs.py:121 UserWarning: OIVD1m contains redundant complexes according to GPR, editing GPR
/data2/hratch/Software/me_analyses_1_env/lib/python3.9/site-packages/human_me/preprocess/correct_inputs.py:121 UserWarning: OIVD2m contains redundant complexes according to GPR, editing GPR
/data2/hratch/Software/me_analyses_1_env/lib/python3.9/site-packages/human_me/preprocess/correct_inputs.py:121 UserWarning: OIVD3m contains redundant complexes according to GPR, editing GPR
/data2/hratch/Software/me_analyses_1_env/lib/python3.9/site-packages/human_me/preprocess/correct_inputs.py:121 UserWarning: PFK contains redundant complexes according to GPR, editing GPR
/data2/hratch/Software/me_analyses_1_env/lib/python3.

Check for the recon2.2 HGNC:HGNC error
Remove genes not participating in reactions


/data2/hratch/Software/me_analyses_1_env/lib/python3.9/site-packages/human_me/preprocess/correct_inputs.py:167 UserWarning: Your metabolic model contains genes with HGNC:HGNC:####, changing to HGNC:####


Build ME Model on full recon2.2:

In [24]:
deg_args = {'reversible_complex_formation': True,
                       'couple': True, 'couple_ribosome': True,
                       'nonenzyme_degradation': False,
                       'complex_degradation': True}

me_model, builder = build_me(model_id = 'full_recon',
                             psim_me=psim_me,
                             me_input_model=me_input_model,
                             deg_args = deg_args,
                             compress_mrna = True,
                             minimal_proteome = False,
                             n_cores = n_cores,
                             check_all = True,
                             seed = 888)

Generate ubiquitin reactions for proteasomal degradation
Generate ribosome
Generate protein expression reactions for metabolic enzymes and non-machinery


100%|█████████████████████████████████████████| 571/571 [00:10<00:00, 55.75it/s]


Express dummy protein
Get metabolic module complex information


100%|██████████████████████████████████████| 4742/4742 [00:07<00:00, 660.67it/s]


Get expression module complex information


...

3556 of 8619 protein degradation reactions will be removed because they are not associated with an active enzyme
Couple enzyme degradation to catalysis


100%|███████████████████████████████████| 99871/99871 [2:12:24<00:00, 12.57it/s]


Add biomass component to reactions
Generate ME-Model
Check reaction mass balances
Make sure all reactions received correct coupled machinery


100%|█████████████████████████████████| 103510/103510 [01:14<00:00, 1383.71it/s]


Add gene objects
Time to build: 158.31 minutes


In [26]:
with urllib.request.urlopen(build_files_url + "required_metabolic_model_metabolites.json") as url:
    required_metabolites = json.loads(url.read().decode())
required_metabolites = flatten_list([v for v in required_metabolites.values()])   

expression_reactions = [r.id for r in me_model.reactions if not hasattr(r, 'cobra_id')]

In [29]:
counsumed_metabolites = []

for m_id in required_metabolites:
    m_reactions = {r.id for r in me_model.metabolites.get_by_id(m_id).reactions}
    m_reactions = list(m_reactions.intersection(expression_reactions))
    
    counter = 0
    for m_reaction in m_reactions:
        substrates = []
        for m, stoich in me_model.reactions.get_by_id(m_reaction).metabolites.items():
            if isinstance(stoich, sympy.Expr):
                stoich_val = float(stoich.subs(params.mu, 1))
            else:
                stoich_val = stoich
            if stoich_val < 0:
                substrates.append(m.id)
        if m_id in substrates:
            counter += 1
    if counter > 0:
        counsumed_metabolites.append(m_id)

In [34]:
print('{} of {} metabolites required by the expression module act as substrates'.format(len(counsumed_metabolites), len(required_metabolites)))

51 of 206 metabolites required by the expression module act as substrates


In [36]:
fn = os.path.join(human_me_data_path, "required_metabolites_consumed_in_Ematrix.txt")
with open(fn, "w") as file:
    for item in counsumed_metabolites:
        file.write(f"{item}\n")